In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import arya

from astropy.table import Table

import pyneb as pn

In [ ]:
dwarf_cat = Table.read("stellar_mass_emline_dwarfs.fits").to_pandas()

In [ ]:
lines = ["oiii4363", "oii3726", "oii3729", "oiii5007", "halpha", "hbeta"]

In [ ]:
filt = np.full(len(dwarf_cat), False)

# at least 1 auroal line
for line in ["oiii4363"]:
    snr = dwarf_cat[line + "_flux"] / dwarf_cat[line + "_fluxerr"]
    filt |= snr > 3

print(np.sum(filt))

for line in ["oii3726", "oii3729", "oiii5007", "hbeta"]:
    snr = dwarf_cat[line + "_flux"] / dwarf_cat[line + "_fluxerr"]
    filt &= snr > 3




print(np.sum(filt))

# quality cuts from Schotle+2026
filt &= dwarf_cat["deltachi2"] >= 40
filt &= dwarf_cat["zcat_primary"] == b't'
filt &= dwarf_cat["spectype"] == b"GALAXY"
filt &= dwarf_cat["zwarn"] == 0

filt &= dwarf_cat["hbeta_ew"] > 20

filt &= dwarf_cat["oiii4363_sigma"] < 2
filt &= (dwarf_cat["halpha_flux"] / dwarf_cat["halpha_fluxerr"] > 3) # or hbeta


print(np.sum(filt))

In [ ]:
good_cat = dwarf_cat[filt].drop_duplicates()

In [ ]:
good_cat

In [ ]:
good_cat["targetid"].values[np.argsort(good_cat["halpha_flux"] / good_cat["halpha_fluxerr"])][-10:-1]

## DiagramsO3.plotGrotrian(tem=1e4, den=1e2, thresh_int=1e-3, unit = 'eV', ax=ax)

In [ ]:
O3 = pn.Atom("O", 3)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 6))

O3.plotGrotrian(tem=1e4, ax=ax)

In [ ]:
O3.plotGrotrian2(tem=1e4, ms=0.3)

In [ ]:
t = np.linspace(5, 30, 100) * 1e3

fig, ax = plt.subplots()

ax.set(xlabel = "temperature / K", ylabel = "log emissivity", xscale="log", 
      title="O III optical lines")


for (i, line) in enumerate([4959, 5007, 4363]):
    y = O3.getEmissivity(t, 1, wave=line)

    ax.plot((t), np.log10(y), label = f"$\\lambda${line}Å")

    y2 = O3.getEmissivity(t, 1e4, wave=line)
    ax.plot(t, np.log10(y2), color=f"C{i}", ls="--")


ax.plot([], [], "k-", label=r"$\rho=1$")
ax.plot([], [], "k--", label=r"$\rho=10^4$")

plt.legend()

In [ ]:
H_balmer = pn.RecAtom("H", 1)

In [ ]:
t = np.linspace(5, 30, 100) * 1e3

fig, ax = plt.subplots()

ax.set(xlabel = "temperature / K", ylabel = r"H$\alpha$ / H$\beta$", xscale="log", 
      title="Balmer ratio")


y = h_balmer.getEmissivity(t, 1, wave=6563)

y0 = h_balmer.getEmissivity(t, 1, wave=4861)

    
ax.plot((t), (y / y0))

y2 = h_balmer.getEmissivity(t, 1e4, wave=6563)
y0 = h_balmer.getEmissivity(t, 1e4, wave=4861)

ax.plot(t, (y2 / y0), color=f"C{i}", ls="--")


ax.plot([], [], "k-", label=r"$\rho=1$")
ax.plot([], [], "k--", label=r"$\rho=10^4$")

plt.legend()


# Emission line fitting

In [ ]:
line_labels = {
    "oii3726": "O2_3726A",
    "oii3729": "O2_3729A",
    "oiii4363": "O3_4363A",
    "oiii4959": "O3_4959A",
    "oiii5007": "O3_5007A",
    "hbeta": "H1r_4861A",
    "halpha": "H1r_6563A",
    "nii6583": "N2_6584A",
}

In [ ]:
def write_observation(catalogue, filename):
    with open(filename, "w") as f:

        lines = line_labels.keys()
        
        f.write("NAME\t")
        for line in lines:
            f.write(line_labels[line] + "\t")
            f.write(line_labels[line] + "_e" + "\t")
        f.write("\n")

        
        for idx, row in catalogue.iterrows():
            f.write(str(row["targetid"]) + "\t")
            for line in lines:
                f.write(str(row[line + "_flux"]))
                f.write("\t")
                f.write(str(row[line + "_fluxerr"]))
                f.write("\t")

            f.write("\n")


In [ ]:
write_observation(good_cat, "good_emlines.tsv")

In [ ]:
obs = pn.Observation("good_emlines.tsv")

In [ ]:
obs.def_EBV()
E_BV = obs.extinction.E_BV

E_BV[E_BV < 0] = 0

obs.extinction.E_BV = E_BV
obs.correctData()

In [ ]:
print(obs.getIntens(obsName=obs.names[1]))
print()

print(obs.getIntens(obsName=obs.names[1], returnObs=True))

In [ ]:
pwd

In [ ]:
obs.lineLabels

### Electron temperature

In [ ]:
diags = pn.Diagnostics()

diags.addDiagsFromObs(obs)

In [ ]:
diags.diags

In [ ]:
# this simply evaluates the flux ratio between these lines

r_oii = diags.eval_diag("[OII] 3726/3729", obs)
r_oiii = diags.eval_diag("[OIII] 4363/5007", obs)

In [ ]:
intens =  obs.getIntens()

In [ ]:
Nobs = len(intens["O3_4363A"])

In [ ]:
Te, Ne = diags.getCrossTemDen("[OIII] 4363/5007", "[OII] 3726/3729", r_oiii, r_oii)

In [ ]:
intens_orig =  obs.getIntens(returnObs=True)

In [ ]:
fig, ax = plt.subplots()

plt.scatter(intens_orig["H1r_6563A"] / intens_orig["H1r_4861A"], obs.extinction.E_BV)

ax.set(
    xlabel = r"H$\alpha$ / H$\beta$",
    ylabel = "E(B-V)"
)

In [ ]:
plt.hist(obs.extinction.E_BV)

In [ ]:
plt.hist(Te)

In [ ]:
fig, ax = plt.subplots()

plt.scatter(intens["O3_4363A"] / intens["O3_5007A"] , Te / 1e4)

ax.set(
    xlabel = r"[O III] 4363 / [O III] 5007",
    ylabel = "$T_e / 10^4 K$",
    # xscale = "log", 
    # yscale = "log"
)

In [ ]:
fig, ax = plt.subplots()

plt.scatter(intens["O2_3726A"] / intens["O2_3729A"] , Ne / 100)

ax.set(
    xlabel = r"[O II] 3726 / [O II] 3729",
    ylabel = "$n_e / 10^2$ cm$^{-2}$",
    # xscale = "log", 
    # yscale = "log"
)

In [ ]:
plt.hist(np.log10(Ne))

In [ ]:
all_atoms = pn.getAtomDict(atom_list=obs.getUniqueAtoms())

In [ ]:
ion_abund = {}


for line in obs.getSortedLines():
    if line.atom == 'H1r':
        continue

    print(line)

    ion_abund[line.label] = all_atoms[line.atom].getIonAbundance(line.corrIntens, Te, Ne, 
        to_eval = line.to_eval, Hbeta=intens["H1r_4861A"])
    

In [ ]:
for line in  ion_abund:
    mean = np.nanmean(np.asarray(ion_abund[line]))
    print('{:16s}: {:4.6f}'.format(str(line), 12+np.log10(mean)))

In [ ]:
eg = pn.EmisGrid("O", 3)

In [ ]:
O2 = pn.Atom("O", 2)

In [ ]:
fig, ax = plt.subplots()
plt.scatter(
    np.log10(intens["O2_3726A"] / intens["H1r_4861A"]),
    np.log10(ion_abund["O2_3726A"]), 
    c=Te, 
    s=1
)
plt.colorbar(label = "Te")

for T in [1e4, 1.4e4, 2e4]:
    x_model = np.linspace(-1, 0, 100)
    n_e = 100
    y_model = O2.getIonAbundance(10**x_model, T, n_e, wave=3726, Hbeta=1)
    plt.plot(x_model, np.log10(y_model))
    # plt.plot(x_model, np.log10(y_model[0]) + x_model - x_model[0])


ax.set(
    xlabel = r"log [O II] 3726 / H$\beta$",
    ylabel = r"log O+ / H",
)

In [ ]:
fig, ax = plt.subplots()
plt.scatter(
    np.log10(intens["O3_5007A"] / intens["H1r_4861A"]),
    np.log10(ion_abund["O3_5007A"]), 
    c=Te, 
    s=1
)
plt.colorbar(label = "Te")

for T in [1e4, 1.4e4, 2e4]:
    x_model = np.linspace(0.5, 1, 100)
    n_e = 100
    y_model = O3.getIonAbundance(10**x_model, T, n_e, wave=5007, Hbeta=1)
    plt.plot(x_model, np.log10(y_model))
    # plt.plot(x_model, np.log10(y_model[0]) + x_model - x_model[0])


ax.set(
    xlabel = r"log [O III] 5007 / H$\beta$",
    ylabel = r"log O++ / H",
)

In [ ]:
plt.scatter(ion_abund["O2_3726A"], ion_abund["O2_3729A"] / ion_abund["O2_3726A"])

In [ ]:
plt.scatter(ion_abund["O3_4363A"], ion_abund["O3_5007A"] / ion_abund["O3_4363A"])

In [ ]:
OII_H = (ion_abund["O2_3726A"] + ion_abund["O2_3729A"]) / 2
OIII_H = (ion_abund["O3_4363A"] + ion_abund["O3_5007A"]) / 2


In [ ]:
O_H = OII_H + OIII_H

In [ ]:
eps_o = np.log10(O_H) + 12

In [ ]:
plt.hist(eps_o, bins=20)

In [ ]:
good_cat["eps_o"] = eps_o

In [ ]:
R23 = (intens["O3_5007A"] + intens["O3_4959A"] + intens["O2_3726A"]) / intens["H1r_4861A"]

In [ ]:
y = np.log10(intens["O2_3726A"] / (intens["O3_4959A"] + intens["O3_5007A"]))
plt.hist(y)

In [ ]:
plt.scatter(np.log10(R23), y, s=1)

In [ ]:
y = np.median(y)
y = 1

In [ ]:
fig, ax = plt.subplots()


plt.scatter(np.log10(R23), eps_o, s=1, lw=0)

y=0
x_model = np.linspace(0.6, 0.9, 1000)
y_model = 6.486 + 1.401 * x_model

plt.plot(x_model, y_model, color="C1")

ax.set(
    xlabel = "log R23",
    ylabel = r"12 + log(O/H) ($T_e$)"
)


In [ ]:
N2 = intens["N2_6584A"] / intens["H1r_6563A"]

In [ ]:
fig, ax = plt.subplots()

plt.scatter(np.log10(N2), eps_o, lw=0, s=1)

x_model = np.linspace(-2.5, -1, 100)
y_model = 9.263 + 0.836*x_model
plt.plot(x_model, y_model, color="C1")

ax.set(
    xlabel = r"$\log$ N2 = $\log($[N II] 6583 / H$\alpha$)",
    ylabel = r"$12 + \log(\textrm{O/H})$ ($T_e$)"
)

In [ ]:
good_cat.sort_values("eps_o")[["targetid", "ra", "dec", "eps_o"]]

In [ ]:
Z_SUN = 0.016

In [ ]:
fig, ax = plt.subplots()

plt.scatter(good_cat.eps_o - 8.7, np.log10(good_cat.z_cg / Z_SUN), s=1)

plt.xlabel(r"[O/H] ($T_e$)")
plt.ylabel(r"$\log Z / Z_\odot$ (CIGALE)")
plt.plot([-1.5, 0], [-1.5, 0])

ax.set(aspect=1)

In [ ]:
fig, ax = plt.subplots()

plt.scatter(good_cat.logm_cigale, good_cat.eps_o, s=1, c=np.log10(good_cat.sfr) - good_cat.logm_cigale)

ax.set(
    xlabel = r"$\log M_\star / \text{M}_\odot$",
    ylabel = r"$12 + \log(\mathrm{O/H})$",
    xlim = (5.5, 9.5)
)

In [ ]:
fig, ax = plt.subplots()

plt.scatter(np.log10(good_cat.sfr.values) - good_cat.logm_cigale + 9, good_cat.eps_o, s=1)

ax.set(
    xlabel = r"$\log$ sSFR",
    ylabel = r"$12 + \log(\mathrm{O/H})$",
    xlim=(0, 2)
)

In [ ]:
from astropy.cosmology import Planck18

In [ ]:
cosmo = Planck18

In [ ]:
cosmo

In [ ]:
from astropy import units as u

In [ ]:
flux_scale = np.array(good_cat["flux_scale"])

In [ ]:
F_Ha = flux_scale * np.array(intens["H1r_6563A"]) * 1e-17  * u.erg / u.s / u.cm**2 # units wrong in documentation?

In [ ]:
F_Ha

In [ ]:
dL = cosmo.luminosity_distance(good_cat["z"])
LHa = 4*np.pi * dL**2 * F_Ha
sfr = 7.9e-42 * LHa * u.M_sun /u.yr / u.erg * u.s * 0.63 

In [ ]:
sfr = sfr.to("M_sun/yr")

In [ ]:
fig, ax = plt.subplots()

plt.scatter(np.log10(good_cat.sfr_cg), np.log10(sfr.value), s=1, lw=0)
# plt.xlim(6, 10)
# plt.ylim(2.5, 5.5)

plt.plot([-2, 2], [-2, 2])
plt.ylim(-2, 2)
plt.xlim(-2, 2)

ax.set(
    xlabel = "log SFR CIGALE",
    ylabel = "log SFR Halpha",
    aspect=1
)

In [ ]:
sui_table = Table.read("sui+26_table2.fits")

In [ ]:
from astropy.table import join

In [ ]:
good_cat["TARGETID"] = good_cat.targetid
good_cat["sfr"] = sfr.value

In [ ]:
good_cat.sfr

In [ ]:
matched = join(Table.from_pandas(good_cat), sui_table, "TARGETID")

In [ ]:
matched

In [ ]:
fig, ax = plt.subplots()
plt.scatter(matched["eps_o"], matched["O_ABUNDANCE"])

lims = (7.4, 7.7)
ax.set(
    xlabel = "12 + log O/H    me",
    ylabel = "12 + log O/H    Sui+26",
    aspect=1, 
    xlim = lims, 
    ylim = lims)

plt.plot(lims, lims, "k-")

In [ ]:
fig, ax = plt.subplots()
plt.scatter(np.log10(matched["sfr"]), matched["logSFR"] - np.log10(matched["sfr"]))

lims = (-2, 1)
ax.set(
    xlabel = "log SFR    me",
    ylabel = "log SFR Sui+26",
    xlim = lims, 
    # ylim = (-0.1, 0.1)
)


In [ ]:
dwarf_cat["TARGETID"] = dwarf_cat.targetid

dwarf_tab = Table.from_pandas(dwarf_cat)

In [ ]:
all_matches = join(sui_table, dwarf_tab, "TARGETID")

In [ ]:
plt.hist(all_matches["oiii4363_flux"] / all_matches["oiii4363_fluxerr"])

In [ ]:
fig, ax = plt.subplots()

plt.scatter(good_cat.logm_cigale, good_cat.eps_o, s=1,)

plt.scatter(sui_table["mass"], sui_table["O_ABUNDANCE"], s=0.9)
ax.set(
    xlabel = r"$\log M_\star / \text{M}_\odot$",
    ylabel = r"$12 + \log(\mathrm{O/H})$",
    xlim = (5.5, 11)
)

In [ ]:
low_z = good_cat[good_cat.eps_o < 7.69]

In [ ]:
plt.scatter(dwarf_cat.ra, dwarf_cat.z, s=0.1, lw=0, color="k", alpha=0.1)

plt.ylim(0, 0.2)

In [ ]:
plt.scatter(good_cat.ra, good_cat.z, s=1, lw=0, color="k", alpha=1)
plt.scatter(low_z.ra, low_z.z, s=5, lw=0, color="C1", alpha=1)

plt.ylim(0, 0.2)

In [ ]:
plt.scatter(good_cat.ra, good_cat.z, s=1, lw=0, color="k", alpha=0.1)

plt.ylim(0, 0.2)